Requirement
  1. Load data from flight-time.json into a table
  2. Table structure is given below

    FL_DATE DATE, 
    OP_CARRIER STRING, 
    OP_CARRIER_FL_NUM STRING, 
    ORIGIN STRING, 
    ORIGIN_CITY_NAME STRING, 
    DEST STRING, 
    DEST_CITY_NAME STRING, 
    CRS_DEP_TIME LONG, 
    DEP_TIME LONG, 
    WHEELS_ON INT, 
    TAXI_IN INT, 
    CRS_ARR_TIME LONG, 
    ARR_TIME LONG, 
    CANCELLED INT, 
    DISTANCE INT

1. Read data from the flight-time.json file

In [0]:
flight_time_raw_df = (
    spark.read.format("json")
    .option("mode","FAILFAST")
    .option("dataframe","M/d/yyyy")
    .load("/Volumes/dev_catalog/spark_db/datasets/spark_programming/data/flight-time.json")
)

2. Investigate the dataframe data and schema for problems

In [0]:
flight_time_raw_df.limit(3).display()
flight_time_raw_df.printSchema()

3. Read datafile with schema-on-read

In [0]:
flight_time_raw_df = (
    spark.read.format("json")
    .option("mode","FAILFAST")
    .option("dateformat","M/d/yyyy")
    .load("/Volumes/dev_catalog/spark_db/datasets/spark_programming/data/flight-time.json")
)

4. Investigate the Dataframe data and schema for problems

In [0]:
flight_time_raw_df.display()


## Point 
Table creation and defining schema 

In [0]:
%sql
-- What kind of schema is this?
-- Table (metastore) schema
-- Stored in Hive Metastore / Unity Catalog
-- Exists even if no Spark job is running

create table if not exists dev_catalog.spark_db.flight_time_raw(
  FL_DATE DATE, OP_CARRIER STRING, OP_CARRIER_FL_NUM STRING, ORIGIN STRING, ORIGIN_CITY_NAME STRING, DEST STRING, DEST_CITY_NAME STRING, CRS_DEP_TIME LONG, DEP_TIME LONG, WHEELS_ON INT, TAXI_IN INT, CRS_ARR_TIME LONG, ARR_TIME LONG, CANCELLED INT, DISTANCE INT
)

## Point
Matching datatypes of datafram with table schema by tranforming column data through withcolumn() function before entering data into table from dataframe.

In [0]:
from pyspark.sql.functions import *

flight_time_raw_new_df = (flight_time_raw_df
.withColumn("FL_DATE", to_date("FL_DATE", "M/d/yyyy"))
.withColumn("OP_CARRIER_FL_NUM", col("OP_CARRIER_FL_NUM").cast("string"))
.withColumn("WHEELS_ON", col("WHEELS_ON").cast("int"))
.withColumn("TAXI_IN", col("TAXI_IN").cast("int"))
.withColumn("CANCELLED", col("CANCELLED").cast("int"))
.withColumn("DISTANCE", col("DISTANCE").cast("int"))
)
flight_time_raw_new_df.printSchema()

## Point
Data inserted from from into table successfully.

In [0]:
flight_time_raw_new_df.write.mode("overwrite").saveAsTable("dev_catalog.spark_db.flight_time_raw")
spark.sql("select count(*) from dev_catalog.spark_db.flight_time_raw").display()


5.  Define Dataframe schema before reading it.
             
             Other way to do samethings

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField

flight_schema = StructType([
    StructField("FL_DATE", DateType()),
    StructField("OP_CARRIER", StringType()),
    StructField("OP_CARRIER_FL_NUM", StringType()),
    StructField("ORIGIN", StringType()),
    StructField("ORIGIN_CITY_NAME", StringType()),
    StructField("DEST", StringType()),
    StructField("DEST_CITY_NAME", StringType()),
    StructField("CRS_DEP_TIME", LongType()),
    StructField("DEP_TIME", LongType()),
    StructField("WHEELS_ON", IntegerType()),
    StructField("TAXI_IN", IntegerType()),
    StructField("CRS_ARR_TIME", LongType()),
    StructField("ARR_TIME", LongType()),
    StructField("CANCELLED", IntegerType()),
    StructField("DISTANCE", IntegerType())
])

6. Read datafile with schema-on-read

In [0]:
# Point to be noticed defined 
# .schema(flight_schema)
# which feed schema to  flight_time_dataframe at runtime at job-level only for that DataFrame, not permanently.

flight_time_dataframe = (
    spark.read
        .format("json")
        .option("mode", "FAILFAST")
        .option("dateFormat", "M/d/yyyy")
        .schema(flight_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/flight-time.json")
)

7. Save the Dataframe to the table flight_time_raw

In [0]:
# Giving Error 
flight_time_dataframe.write.mode("overwrite").saveAsTable("dev_catalog.spark_db.flight_two_time_raw")